In [5]:
# [1] 설치
# 최초 1회만 필요
# pip install fin:contentReference[oaicite:0]{index=0} pandas numpy requests lxml beautifulsoup4 html5lib

from datetime import datetime
from pathlib import Path
from io import StringIO
import time
import re

import numpy as np
import pandas as pd
import yfinance as yf
import FinanceDataReader as fdr
import requests


# [2] 기본 설정

START_DATE = "2020-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

OUT_DIR = Path("./market_agent_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PRICE = OUT_DIR / "market_price_data.csv"
OUT_FLOW = OUT_DIR / "market_flow_data.csv"

KR_STOCKS = {
    "005930": "Samsung Electronics",
    "000660": "SK Hynix",
    "042700": "Hanmi Semiconductor",
}

KR_INDEXES = {
    "KS11": "KOSPI",
    "KQ11": "KOSDAQ",
}

GLOBAL_ASSETS = {
    "SOXX": ("iShares Semiconductor ETF", "US_ETF"),
    "SMH": ("VanEck Semiconductor ETF", "US_ETF"),
    "^IXIC": ("NASDAQ Composite", "US_INDEX"),
    "^GSPC": ("S&P 500", "US_INDEX"),
    "KRW=X": ("USD/KRW", "FX"),
}


# [3] 공통 정리 함수

def clean_number(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = x.replace(",", "")
    x = x.replace("%", "")
    x = x.replace("+", "")
    x = x.replace("−", "-")
    x = x.replace("▲", "")
    x = x.replace("▼", "-")
    x = re.sub(r"[^0-9.\-]", "", x)

    if x in ["", "-", "."]:
        return np.nan

    try:
        return float(x)
    except:
        return np.nan


def to_long_price_df(df, asset_code, asset_name, market, source):
    columns = [
        "Date", "AssetCode", "AssetName", "Market", "Source",
        "Open", "High", "Low", "Close", "Volume", "Value"
    ]

    if df is None or df.empty:
        return pd.DataFrame(columns=columns)

    out = df.copy().reset_index()

    if "Date" not in out.columns:
        out = out.rename(columns={out.columns[0]: "Date"})

    for col in ["Open", "High", "Low", "Close", "Volume"]:
        if col not in out.columns:
            out[col] = np.nan

    if "Value" not in out.columns:
        out["Value"] = np.nan

    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out["AssetCode"] = asset_code
    out["AssetName"] = asset_name
    out["Market"] = market
    out["Source"] = source

    out = out[columns].copy()
    out = out.dropna(subset=["Date"])
    out = out.sort_values("Date").reset_index(drop=True)

    return out


# [4] 국내 주식 가격 수집

def fetch_fdr_kr_stock_prices(kr_stocks):
    frames = []

    for ticker, asset_name in kr_stocks.items():
        try:
            raw = fdr.DataReader(ticker, START_DATE, END_DATE)

            price_df = to_long_price_df(
                raw,
                asset_code=ticker,
                asset_name=asset_name,
                market="KR_STOCK",
                source="FinanceDataReader"
            )

            frames.append(price_df)
            print(f"[OK] FDR stock: {ticker}, rows={len(price_df)}")

        except Exception as e:
            print(f"[ERROR] FDR stock 실패: {ticker} / {e}")

        time.sleep(0.2)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# [5] 국내 지수 가격 수집

def fetch_fdr_kr_index_prices(kr_indexes):
    frames = []

    for code, name in kr_indexes.items():
        try:
            raw = fdr.DataReader(code, START_DATE, END_DATE)

            price_df = to_long_price_df(
                raw,
                asset_code=code,
                asset_name=name,
                market="KR_INDEX",
                source="FinanceDataReader"
            )

            frames.append(price_df)
            print(f"[OK] FDR index: {code}, rows={len(price_df)}")

        except Exception as e:
            print(f"[ERROR] FDR index 실패: {code} / {e}")

        time.sleep(0.2)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# [6] 해외 가격 수집

def fetch_yfinance_global_prices(global_assets):
    frames = []

    for ticker, (asset_name, market) in global_assets.items():
        try:
            raw = yf.download(
                ticker,
                start=START_DATE,
                end=END_DATE,
                auto_adjust=False,
                progress=False
            )

            if raw is None or raw.empty:
                print(f"[WARN] yfinance 빈 데이터: {ticker}")
                continue

            if isinstance(raw.columns, pd.MultiIndex):
                raw.columns = raw.columns.get_level_values(0)

            keep_cols = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in raw.columns]
            raw = raw[keep_cols].copy()

            price_df = to_long_price_df(
                raw,
                asset_code=ticker,
                asset_name=asset_name,
                market=market,
                source="yfinance"
            )

            frames.append(price_df)
            print(f"[OK] yfinance: {ticker}, rows={len(price_df)}")

        except Exception as e:
            print(f"[ERROR] yfinance 실패: {ticker} / {e}")

        time.sleep(0.2)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# [7] 네이버 금융 외국인/기관 일별 매매동향 수집
# 주의: 네이버 금융에서 얻는 값은 순매수 "금액"이 아니라 주로 순매수 "수량"입니다.
# 따라서 NetBuyVolume 컬럼으로 저장합니다.

def fetch_naver_flow_one_stock(ticker, asset_name, max_pages=250):
    columns = [
        "Date", "AssetCode", "AssetName", "Market", "Source",
        "InvestorType", "NetBuyVolume"
    ]

    frames = []
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    for page in range(1, max_pages + 1):
        url = f"https://finance.naver.com/item/frgn.naver?code={ticker}&page={page}"

        try:
            res = requests.get(url, headers=headers, timeout=10)
            res.raise_for_status()

            tables = pd.read_html(StringIO(res.text), encoding="cp949")

            target = None
            for table in tables:
                cols = [str(c).strip() for c in table.columns]
                if "날짜" in cols and ("기관" in cols or "외국인" in cols):
                    target = table.copy()
                    break

            if target is None or target.empty:
                continue

            target.columns = [str(c).strip() for c in target.columns]
            target = target.dropna(how="all")

            if "날짜" not in target.columns:
                continue

            target["Date"] = pd.to_datetime(target["날짜"], errors="coerce")
            target = target.dropna(subset=["Date"])

            if target.empty:
                continue

            target = target[target["Date"] >= pd.to_datetime(START_DATE)]
            target = target[target["Date"] <= pd.to_datetime(END_DATE)]

            if target.empty:
                last_date_candidates = pd.to_datetime(table["날짜"], errors="coerce") if "날짜" in table.columns else pd.Series([])
                if len(last_date_candidates.dropna()) > 0:
                    oldest = last_date_candidates.dropna().min()
                    if oldest < pd.to_datetime(START_DATE):
                        break
                continue

            page_rows = []

            if "외국인" in target.columns:
                temp = target[["Date", "외국인"]].copy()
                temp["InvestorType"] = "Foreign"
                temp["NetBuyVolume"] = temp["외국인"].apply(clean_number)
                page_rows.append(temp[["Date", "InvestorType", "NetBuyVolume"]])

            if "기관" in target.columns:
                temp = target[["Date", "기관"]].copy()
                temp["InvestorType"] = "Institution"
                temp["NetBuyVolume"] = temp["기관"].apply(clean_number)
                page_rows.append(temp[["Date", "InvestorType", "NetBuyVolume"]])

            if page_rows:
                page_df = pd.concat(page_rows, ignore_index=True)
                page_df["AssetCode"] = ticker
                page_df["AssetName"] = asset_name
                page_df["Market"] = "KR_STOCK"
                page_df["Source"] = "NaverFinance"
                page_df = page_df[columns]
                frames.append(page_df)

        except Exception as e:
            print(f"[WARN] Naver flow page 실패: {ticker}, page={page}, error={e}")

        time.sleep(0.15)

    if not frames:
        print(f"[WARN] Naver flow 빈 데이터: {ticker}")
        return pd.DataFrame(columns=columns)

    result = pd.concat(frames, ignore_index=True)
    result = result.dropna(subset=["Date"])
    result = result.drop_duplicates(subset=["Date", "AssetCode", "InvestorType"])
    result = result.sort_values(["Date", "AssetCode", "InvestorType"]).reset_index(drop=True)

    print(f"[OK] Naver flow: {ticker}, rows={len(result)}")
    return result


def fetch_naver_flows(kr_stocks, max_pages=250):
    frames = []

    for ticker, asset_name in kr_stocks.items():
        flow_df = fetch_naver_flow_one_stock(ticker, asset_name, max_pages=max_pages)
        if not flow_df.empty:
            frames.append(flow_df)

    columns = [
        "Date", "AssetCode", "AssetName", "Market", "Source",
        "InvestorType", "NetBuyVolume"
    ]

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)


# [8] 실행

print("[START] Market Agent data collection")

kr_stock_price_df = fetch_fdr_kr_stock_prices(KR_STOCKS)
kr_index_price_df = fetch_fdr_kr_index_prices(KR_INDEXES)
global_price_df = fetch_yfinance_global_prices(GLOBAL_ASSETS)
flow_df = fetch_naver_flows(KR_STOCKS, max_pages=250)

price_df = pd.concat(
    [kr_stock_price_df, kr_index_price_df, global_price_df],
    ignore_index=True
)

price_df["Date"] = pd.to_datetime(price_df["Date"], errors="coerce")
price_df = price_df.dropna(subset=["Date"])
price_df = price_df.sort_values(["Date", "AssetCode"]).drop_duplicates().reset_index(drop=True)

flow_df["Date"] = pd.to_datetime(flow_df["Date"], errors="coerce")
flow_df = flow_df.dropna(subset=["Date"])
flow_df = flow_df.sort_values(["Date", "AssetCode", "InvestorType"]).drop_duplicates().reset_index(drop=True)

price_df.to_csv(OUT_PRICE, index=False, encoding="utf-8-sig")
flow_df.to_csv(OUT_FLOW, index=False, encoding="utf-8-sig")

print("[DONE] Saved files")
print("price:", OUT_PRICE, price_df.shape)
print("flow :", OUT_FLOW, flow_df.shape)


# [9] 결과 점검

print("\n[PRICE COUNT]")
price_count = (
    price_df.groupby(["AssetCode", "AssetName", "Market", "Source"])
    .size()
    .reset_index(name="RowCount")
    .sort_values(["Market", "AssetCode"])
    .reset_index(drop=True)
)
display(price_count)

print("\n[FLOW COUNT]")
if not flow_df.empty:
    flow_count = (
        flow_df.groupby(["AssetCode", "AssetName", "InvestorType"])
        .size()
        .reset_index(name="RowCount")
        .sort_values(["AssetCode", "InvestorType"])
        .reset_index(drop=True)
    )
    display(flow_count)
else:
    print("flow_df is empty")

print("\n[PRICE HEAD]")
display(price_df.head(10))

print("\n[FLOW HEAD]")
display(flow_df.head(10))

[START] Market Agent data collection
[OK] FDR stock: 005930, rows=1550
[OK] FDR stock: 000660, rows=1550
[OK] FDR stock: 042700, rows=1550
[OK] FDR index: KS11, rows=1550
[OK] FDR index: KQ11, rows=1550
[OK] yfinance: SOXX, rows=1586
[OK] yfinance: SMH, rows=1586
[OK] yfinance: ^IXIC, rows=1586
[OK] yfinance: ^GSPC, rows=1586
[OK] yfinance: KRW=X, rows=1644
[WARN] Naver flow 빈 데이터: 005930
[WARN] Naver flow 빈 데이터: 000660
[WARN] Naver flow 빈 데이터: 042700
[DONE] Saved files
price: market_agent_data\market_price_data.csv (15738, 11)
flow : market_agent_data\market_flow_data.csv (0, 7)

[PRICE COUNT]


,AssetCode,AssetName,Market,Source,RowCount
0,KRW=X,USD/KRW,FX,yfinance,1644
1,KQ11,KOSDAQ,KR_INDEX,FinanceDataReader,1550
2,KS11,KOSPI,KR_INDEX,FinanceDataReader,1550
3,000660,SK Hynix,KR_STOCK,FinanceDataReader,1550
4,005930,Samsung Electronics,KR_STOCK,FinanceDataReader,1550
5,042700,Hanmi Semiconductor,KR_STOCK,FinanceDataReader,1550
6,SMH,VanEck Semiconductor ETF,US_ETF,yfinance,1586
7,SOXX,iShares Semiconductor ETF,US_ETF,yfinance,1586
8,^GSPC,S&P 500,US_INDEX,yfinance,1586
9,^IXIC,NASDAQ Composite,US_INDEX,yfinance,1586



[FLOW COUNT]
flow_df is empty

[PRICE HEAD]


,Date,AssetCode,AssetName,Market,Source,Open,High,Low,Close,Volume,Value
0,2020-01-01,KRW=X,USD/KRW,FX,yfinance,1154.400024,1154.400024,1145.449951,1153.750000,0,NaN
1,2020-01-02,000660,SK Hynix,KR_STOCK,FinanceDataReader,96000.000000,96200.000000,94100.000000,94700.000000,2342070,NaN
2,2020-01-02,005930,Samsung Electronics,KR_STOCK,FinanceDataReader,55500.000000,56000.000000,55000.000000,55200.000000,12993228,NaN
3,2020-01-02,042700,Hanmi Semiconductor,KR_STOCK,FinanceDataReader,4055.000000,4125.000000,3970.000000,3980.000000,366562,NaN
4,2020-01-02,KQ11,KOSDAQ,KR_INDEX,FinanceDataReader,672.530000,674.300000,666.620000,674.020000,783730760,NaN
5,2020-01-02,KRW=X,USD/KRW,FX,yfinance,1153.959961,1160.189941,1145.199951,1153.969971,0,NaN
6,2020-01-02,KS11,KOSPI,KR_INDEX,FinanceDataReader,2201.210000,2202.320000,2171.840000,2175.170000,494677752,NaN
7,2020-01-02,SMH,VanEck Semiconductor ETF,US_ETF,yfinance,71.894997,72.470001,71.654999,72.339996,5200400,NaN
8,2020-01-02,SOXX,iShares Semiconductor ETF,US_ETF,yfinance,84.753334,85.430000,84.333336,85.430000,1275300,NaN
9,2020-01-02,^GSPC,S&P 500,US_INDEX,yfinance,3244.669922,3258.139893,3235.530029,3257.850098,3459930000,NaN



[FLOW HEAD]


,Date,AssetCode,AssetName,Market,Source,InvestorType,NetBuyVolume
